In [21]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


---

In [22]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [23]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

In [24]:
from ldpc.bp_decoder import BpDecoder

---

In [25]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [26]:
decoder = BpDecoder(H, schedule="cluster")

In [27]:
n = 486 # length of message
n_frames = 1000
max_iter = 5

message = np.random.randint(0, 2, (n_frames, n))
print("Message shape:", message.shape)

Message shape: (1000, 486)


In [28]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (1000, 648)


In [29]:
m, _ = H.shape
arr = np.arange(m)
schedule = arr.reshape(6, -1)


In [ ]:
snrs = [0, 1, 2, 3, 4, 5, 6, 7]
bers = []

for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        for iter in range(max_iter):
            for cluster in schedule:
                llr = decoder.decode_cluster(cluster)
        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR 0 dB: 0.1579485596707819
BER at SNR 1 dB: 0.13046502057613169
BER at SNR 2 dB: 0.1015082304526749
BER at SNR 3 dB: 0.0709238683127572
BER at SNR 4 dB: 0.025868312757201646
BER at SNR 5 dB: 0.0007119341563786008
BER at SNR 6 dB: 3.08641975308642e-05
